# Gold Hiring Activity Mart

This notebook creates the sector-level hiring activity data mart with daily granularity.

## Architecture

**Input**: `workspace.warehouse.fact_job_postings`  
**Output**: `workspace.gold.gold_hiring_activity`  
**Mode**: Full refresh (CREATE OR REPLACE)

## Schema

**Grain**: Sector × Date  
**Primary Key**: (sector_sk, hiring_date_sk)  
**Partitioning**: sector_sk

**Key Metrics**:
- `total_jobs`: Total jobs in sector
- `new_jobs`: New postings
- `top_role`: Most hired role
- `avg_salary`: Average salary

In [0]:
# Databricks notebook parameters
dbutils.widgets.text("lookback_days", "365", "Lookback Days")
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")

# Get parameter values
lookback_days = int(dbutils.widgets.get("lookback_days"))
force_full_refresh = dbutils.widgets.get("force_full_refresh").lower() == "true"

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# Configuration
CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
GOLD_SCHEMA = f"{CATALOG}.gold"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Current run metadata
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Run ID: {run_id}")
print(f"Lookback days: {lookback_days}")
print(f"Cutoff date: {cutoff_date}")
print(f"Force full refresh: {force_full_refresh}")

In [0]:
# Create metadata table to track refresh runs
metadata_table = f"{METADATA_SCHEMA}.gold_hiring_activity_refresh_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  run_id STRING,
  run_timestamp TIMESTAMP,
  lookback_days INT,
  cutoff_date INT,
  rows_created BIGINT,
  unique_sectors BIGINT,
  unique_dates BIGINT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DOUBLE,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks refresh runs for gold_hiring_activity'
""")

print(f"Metadata table ready: {metadata_table}")

In [0]:
import time

start_time = time.time()

print("Creating gold_hiring_activity mart...")

# Execute CREATE OR REPLACE
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_SCHEMA}.gold_hiring_activity
USING DELTA
PARTITIONED BY (sector_sk)
COMMENT 'Hiring trends and analysis across sectors'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS

WITH daily_sector AS (
  SELECT
    f.sector_sk,
    f.posting_date_sk AS hiring_date_sk,
    f.role_sk,
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS total_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs
  FROM {WAREHOUSE_SCHEMA}.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), {lookback_days}), 'yyyyMMdd') AS INT)
    AND f.sector_sk IS NOT NULL
  GROUP BY f.sector_sk, f.posting_date_sk, f.role_sk
),

top_roles AS (
  SELECT
    sector_sk,
    hiring_date_sk,
    role_sk,
    SUM(total_jobs) AS role_jobs,
    ROW_NUMBER() OVER (PARTITION BY sector_sk, hiring_date_sk ORDER BY SUM(total_jobs) DESC, role_sk) AS rn
  FROM daily_sector
  GROUP BY sector_sk, hiring_date_sk, role_sk
),

aggregated AS (
  SELECT
    ds.sector_sk,
    ds.hiring_date_sk,
    SUM(ds.total_jobs) AS total_jobs,
    SUM(ds.new_jobs) AS new_jobs,
    FIRST(dr.role_name) AS top_role
  FROM daily_sector ds
  LEFT JOIN top_roles tr ON ds.sector_sk = tr.sector_sk AND ds.hiring_date_sk = tr.hiring_date_sk AND tr.rn = 1
  LEFT JOIN {WAREHOUSE_SCHEMA}.dim_role dr ON tr.role_sk = dr.role_sk
  GROUP BY ds.sector_sk, ds.hiring_date_sk
)

SELECT
  sector_sk,
  hiring_date_sk,
  total_jobs,
  new_jobs,
  top_role,
  CAST(NULL AS DECIMAL(15,2)) AS avg_salary,
  CURRENT_TIMESTAMP() AS updated_at
FROM aggregated
ORDER BY hiring_date_sk DESC, sector_sk
""")

processing_time = time.time() - start_time
print(f"✓ Table created successfully in {processing_time:.2f} seconds")

In [0]:
# Get table statistics
table_stats = spark.sql(f"""
SELECT
  COUNT(*) AS rows_created,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT hiring_date_sk) AS unique_dates
FROM {GOLD_SCHEMA}.gold_hiring_activity
""").collect()[0]

rows_created = table_stats["rows_created"]
unique_sectors = table_stats["unique_sectors"]
unique_dates = table_stats["unique_dates"]

# Log metadata - match existing table schema
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("run_timestamp", TimestampType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

metadata_df = spark.createDataFrame([(
    run_id,
    run_timestamp,
    0,  # rows_updated (not applicable for full refresh)
    rows_created,  # rows_inserted
    "success",
    None
)], schema=metadata_schema)

metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)

print(f"\nRefresh Summary:")
print(f"  Run ID: {run_id}")
print(f"  Rows created: {rows_created:,}")
print(f"  Unique sectors: {unique_sectors}")
print(f"  Unique dates: {unique_dates}")
print(f"  Processing time: {processing_time:.2f}s")

In [0]:
%sql
-- Validation: Summary Statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT hiring_date_sk) AS unique_dates,
  MIN(hiring_date_sk) AS earliest_date,
  MAX(hiring_date_sk) AS latest_date,
  SUM(total_jobs) AS total_jobs,
  SUM(new_jobs) AS total_new_jobs,
  AVG(avg_salary) AS overall_avg_salary,
  MAX(updated_at) AS last_refreshed
FROM workspace.gold.gold_hiring_activity;